# Highway Network Testing

In [6]:
%load_ext autoreload
%autoreload 2
# normally Python imports a file only once, the notebook will keep using the old version until you restart the kernel.

import torch
import math
from main import (get_dataloaders, Net, HighwayLayer, PlainConv,
                  run_experiment, evaluate, DATA_DIR, DEVICE, SEED)

# Data prep
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR)
print("sizes:", len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

images, labels = next(iter(train_loader))
print("batch:", tuple(images.shape), tuple(labels.shape))


# Testing the Highway layer
layer = HighwayLayer(channels=16)
x = torch.randn(8, 16, 32, 32)
y = layer(x)
print("in :", tuple(x.shape), "out:", tuple(y.shape))
print("gate mean at init:", torch.sigmoid(layer.gate(x)).mean().item())
print("params:", sum(p.numel() for p in layer.parameters()))
print()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
sizes: 45000 5000 10000
batch: (128, 3, 32, 32) (128,)
in : (8, 16, 32, 32) out: (8, 16, 32, 32)
gate mean at init: 0.1308787614107132
params: 4640



In [3]:

# Testing the Highway layer
layer = HighwayLayer(channels=16)
x = torch.randn(8, 16, 32, 32)
y = layer(x)
print("in :", tuple(x.shape), "out:", tuple(y.shape))
print("gate mean at init:", torch.sigmoid(layer.gate(x)).mean().item())
print("params:", sum(p.numel() for p in layer.parameters()))
print()

in : (8, 16, 32, 32) out: (8, 16, 32, 32)
gate mean at init: 0.1318034678697586
params: 4640



### Random decisions

In [11]:
# Base line: 5,000 imgs, 10 classes, dataset is balanced = Random guessing accuracy = 1/10 = 0.1
print("baseline accuracy:", 1 / 10)

# The loss looks at the probability the model gives to the true class
# loss = −ln(p_true)
# p = (0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1)      # 10 classes, each 1/10
loss_base = - math.log(0.1)
print("baseline loss:", loss_base)
print()

baseline accuracy: 0.1
baseline loss: 2.3025850929940455



In [12]:
print("=== Highway ===")
for lr in [0.3, 0.1, 0.03, 0.01, 0.003]:
    torch.manual_seed(SEED) # same starting weights for every lr
    model = Net(HighwayLayer, layers_per_stage=3, conv_init=False)
    print(f"\n--- lr = {lr} ---")
    history, best_acc = run_experiment(model, train_loader, val_loader, epochs=3, lr=lr, momentum=0.9, device=DEVICE)
    print("best val acc:", best_acc)
print()

print("=== PlainConv ===")
for lr in [0.3, 0.1, 0.03, 0.01, 0.003]:
    torch.manual_seed(SEED) # same starting weights for every lr
    model = Net(PlainConv, layers_per_stage=3, conv_init=False)
    print(f"\n--- lr = {lr} ---")
    history, best_acc = run_experiment(model, train_loader, val_loader, epochs=3, lr=lr, momentum=0.9, device=DEVICE)
    print("best val acc:", best_acc)

=== Highway ===

--- lr = 0.3 ---
epoch  1 | train loss 1.916 acc 0.266 | val loss 1.587 acc 0.376
epoch  2 | train loss 1.520 acc 0.439 | val loss 1.409 acc 0.477
epoch  3 | train loss 1.357 acc 0.510 | val loss 1.232 acc 0.551
best val acc: 0.5508

--- lr = 0.1 ---
epoch  1 | train loss 1.892 acc 0.277 | val loss 1.716 acc 0.333
epoch  2 | train loss 1.459 acc 0.461 | val loss 1.406 acc 0.494
epoch  3 | train loss 1.249 acc 0.549 | val loss 1.107 acc 0.611
best val acc: 0.611

--- lr = 0.03 ---
epoch  1 | train loss 2.050 acc 0.222 | val loss 1.752 acc 0.314
epoch  2 | train loss 1.683 acc 0.360 | val loss 1.571 acc 0.414
epoch  3 | train loss 1.485 acc 0.445 | val loss 1.433 acc 0.482
best val acc: 0.4824

--- lr = 0.01 ---
epoch  1 | train loss 2.196 acc 0.173 | val loss 2.029 acc 0.233
epoch  2 | train loss 1.932 acc 0.271 | val loss 1.752 acc 0.338
epoch  3 | train loss 1.724 acc 0.348 | val loss 1.667 acc 0.371
best val acc: 0.3706

--- lr = 0.003 ---
epoch  1 | train loss 2.298

Plain Conv doesn't learn at all, at any learning rate. Needs proper, better normalized initialization.


```py
# added in Net Class
for m in self.modules():
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
```

In [13]:
print("=== PlainConv ===")
for lr in [0.3, 0.1, 0.03, 0.01, 0.003]:
    torch.manual_seed(SEED) # same starting weights for every lr
    model = Net(PlainConv, layers_per_stage=3, conv_init=True)
    print(f"\n--- lr = {lr} ---")
    history, best_acc = run_experiment(model, train_loader, val_loader, epochs=3, lr=lr, momentum=0.9, device=DEVICE)
    print("best val acc:", best_acc)

=== PlainConv ===

--- lr = 0.3 ---
epoch  1 | train loss nan acc 0.100 | val loss nan acc 0.100
epoch  2 | train loss nan acc 0.100 | val loss nan acc 0.100
epoch  3 | train loss nan acc 0.100 | val loss nan acc 0.100
best val acc: 0.1

--- lr = 0.1 ---
epoch  1 | train loss 2.302 acc 0.106 | val loss 2.305 acc 0.100
epoch  2 | train loss 2.305 acc 0.100 | val loss 2.304 acc 0.100
epoch  3 | train loss 2.305 acc 0.100 | val loss 2.304 acc 0.100
best val acc: 0.1

--- lr = 0.03 ---
epoch  1 | train loss 2.039 acc 0.201 | val loss 1.911 acc 0.252
epoch  2 | train loss 1.715 acc 0.333 | val loss 1.612 acc 0.374
epoch  3 | train loss 1.519 acc 0.425 | val loss 1.381 acc 0.490
best val acc: 0.4896

--- lr = 0.01 ---
epoch  1 | train loss 1.908 acc 0.275 | val loss 1.662 acc 0.375
epoch  2 | train loss 1.595 acc 0.398 | val loss 1.374 acc 0.485
epoch  3 | train loss 1.367 acc 0.494 | val loss 1.292 acc 0.524
best val acc: 0.5236

--- lr = 0.003 ---
epoch  1 | train loss 1.948 acc 0.259 | va

For `lr=3` it was dead, but with lowering the learning rate we see it come alive.